# Multiverse Hybrid v3.0 — Stage 3 Ticket Filter 2000 v1

Stage 2のPRICE/EVカタログに、結果を見る前に固定した7段階の候補フィルタを適用します。

- RESULT / PAYOUT / Settlement は読みません
- realized ROI は計算しません
- どのprofileが勝つかは選びません
- 全7券種に同一閾値familyを適用します
- 出力は候補密度 / NO-BET率の診断だけです
- 巨大ZIP・自動ダウンロードは作りません

iPhoneでは **ランタイム → すべてのセルを実行** だけで構いません。


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, shutil, hashlib, json

drive.mount('/content/drive')
MY=Path('/content/drive/MyDrive')
REPO=Path('/content/multiverse-research-stage3')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.check_call(['git','clone','--depth','1','https://github.com/fufufu1116/multiverse-research.git',str(REPO)])

EXPECTED={
 'v3/historical_all_market/stage3_ticket_filter_diagnostics_v1.py':'dc509e03b464faca97ac2e242ad7ac4c79f55a3d',
 'v3/historical_all_market/governance/STAGE3_TICKET_FILTER_FAMILY_PREREG_v1.md':'ba4175bb044bcacfa66a7b8d089e92c04762b2e6',
 'v3/historical_all_market/runtime_receipts/ALL_MARKET_STAGE3_TICKET_FILTER_STATIC_SELF_CHECK_v1.json':'b9a831ce205b933fe565cf3d2137a243dd2440c6',
}
for rel,exp in EXPECTED.items():
    obs=subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
    if obs!=exp: raise RuntimeError(f'FAIL-CLOSED Git blob mismatch {rel}: {obs} != {exp}')
print('✅ STAGE3 EXACT CODE / PREREG / SELF-CHECK BINDINGS PASS')

STAGE2_DIR=MY/'MULTIVERSE_ALL_MARKET_STAGE2_PRICE_EV_v1'
CAT=STAGE2_DIR/'DEV2000_ALL_MARKET_PRICE_EV_CATALOG_v1.jsonl'
S2_RECEIPT=STAGE2_DIR/'STAGE2_PRICE_EV_RECEIPT_v1.json'
OUT=MY/'MULTIVERSE_ALL_MARKET_STAGE3_TICKET_FILTER_v1'
OUT.mkdir(parents=True,exist_ok=True)
RACE_COUNTS=OUT/'STAGE3_TICKET_FILTER_RACE_COUNTS_v1.csv'
QUALITY=OUT/'STAGE3_TICKET_FILTER_DIAGNOSTICS_QUALITY_v1.json'
RECEIPT=OUT/'STAGE3_TICKET_FILTER_RECEIPT_v1.json'
LOG=OUT/'STAGE3_TICKET_FILTER_RUN_LOG_v1.txt'

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(1<<20),b''): h.update(c)
    return h.hexdigest()

if not CAT.is_file() or not S2_RECEIPT.is_file():
    raise RuntimeError(f'FAIL-CLOSED Stage2 input missing CAT={CAT.exists()} RECEIPT={S2_RECEIPT.exists()}')
s2=json.loads(S2_RECEIPT.read_text(encoding='utf-8'))
if s2.get('status')!='PASS' or s2.get('catalog_sha256')!='34ad32bed6e8b4d700864c46f4533bef1da254c7d87dc7ffe6ec266fd74530dc':
    raise RuntimeError('FAIL-CLOSED Stage2 receipt/catalog binding is not PASS')
if s2.get('result_access') is not False or s2.get('settlement_access') is not False or s2.get('realized_roi_computed') is not False:
    raise RuntimeError('FAIL-CLOSED Stage2 firewall state')

# Existing PASS fast-path: no 557MB rescan when valid outputs already exist.
existing_ok=False
if RECEIPT.is_file() and QUALITY.is_file() and RACE_COUNTS.is_file():
    try:
        r=json.loads(RECEIPT.read_text(encoding='utf-8'))
        q=json.loads(QUALITY.read_text(encoding='utf-8'))
        existing_ok=(
          r.get('status')=='PASS' and q.get('status')=='PASS' and
          r.get('quality_sha256')==sha256(QUALITY) and
          r.get('race_counts_csv_sha256')==sha256(RACE_COUNTS) and
          q.get('stage2_catalog_sha256')=='34ad32bed6e8b4d700864c46f4533bef1da254c7d87dc7ffe6ec266fd74530dc' and
          q.get('profile_monotonicity_violations')==0 and
          q.get('profile_selection_performed') is False and
          q.get('result_access') is False and q.get('payout_access') is False and q.get('settlement_access') is False and
          q.get('realized_roi_computed') is False and
          q.get('scientific_trial_count')==0 and q.get('ECON_HOLDOUT1000')=='SEALED'
        )
    except Exception:
        existing_ok=False

if existing_ok:
    print('✅ STAGE3 ALREADY PASS — 再計算不要')
    print(RECEIPT.read_text(encoding='utf-8'))
else:
    for p in (RACE_COUNTS,QUALITY,RECEIPT,LOG):
        if p.exists(): p.unlink()
    engine=REPO/'v3/historical_all_market/stage3_ticket_filter_diagnostics_v1.py'
    cmd=['python',str(engine),str(CAT),str(RACE_COUNTS),str(QUALITY)]
    print('▶ STAGE3 ticket-filter diagnostics start (Stage2 557MB one-pass)')
    proc=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    LOG.write_text(proc.stdout,encoding='utf-8')
    print('\n'.join(proc.stdout.splitlines()[-120:]))
    if proc.returncode!=0:
        raise RuntimeError(f'FAIL-CLOSED Stage3 return={proc.returncode}; see {LOG}')
    q=json.loads(QUALITY.read_text(encoding='utf-8'))
    if q.get('status')!='PASS' or q.get('profile_monotonicity_violations')!=0:
        raise RuntimeError('FAIL-CLOSED Stage3 quality gate')
    if q.get('profile_selection_performed') is not False or q.get('market_specific_threshold_tuning_performed') is not False:
        raise RuntimeError('FAIL-CLOSED Stage3 prereg family drift')
    if q.get('result_access') is not False or q.get('payout_access') is not False or q.get('settlement_access') is not False or q.get('realized_roi_computed') is not False:
        raise RuntimeError('FAIL-CLOSED Stage3 outcome firewall')
    receipt={
      'record':'STAGE3_TICKET_FILTER_RECEIPT_v1','status':'PASS',
      'stage3_engine_git_blob':'dc509e03b464faca97ac2e242ad7ac4c79f55a3d',
      'stage3_prereg_git_blob':'ba4175bb044bcacfa66a7b8d089e92c04762b2e6',
      'stage2_catalog_sha256':'34ad32bed6e8b4d700864c46f4533bef1da254c7d87dc7ffe6ec266fd74530dc',
      'race_counts_csv_sha256':sha256(RACE_COUNTS),
      'quality_sha256':sha256(QUALITY),
      'input_rows':q['input_rows'],'unique_races':q['unique_races'],'profile_count':len(q['profiles']),
      'profile_selection_performed':False,'result_access':False,'payout_access':False,'settlement_access':False,
      'realized_roi_computed':False,'scientific_trial_count':0,'ECON_HOLDOUT1000':'SEALED'
    }
    RECEIPT.write_text(json.dumps(receipt,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
    print('✅ STAGE3 PASS')
    print(RECEIPT.read_text(encoding='utf-8'))

print('Drive folder:',OUT)
print('RESULT/PAYOUT/Settlement/realized ROI access = none')
print('Profile promotion = none')
print('ECON_HOLDOUT1000 = SEALED')
